In [2]:
# Runnable 클래스(모든 모델이 상속 받음)
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:1b")

In [21]:
from langchain_core.prompts  import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = PromptTemplate(
    template="What is the capital of {country}? Return the name of the capital only",
    input_variables=["country"]
)

prompt = prompt_template.invoke({"country": "France"})
ai_response = llm.invoke(prompt)

output_parser = StrOutputParser()

# 여기보세요
answer = output_parser.invoke(llm.invoke(prompt_template.invoke({"country": "France"})))
# 기능이 다른데 다 invoke를 함(결과 형식, LLM답변, prompt)
# 그리고 도구도 invoke를 함
# 확인을 위해 클래스를 타고 들어가보자
# 그럼 모두 class Runnable(ABC, Generic[Input, Output])를 상속 받음
# 그리고 Runnable에 invoke 메서드가 있음 -> LangChain은 결국 실행가능한 무언가의 조합

In [22]:
# 다 Runnable이라서 이렇게 파이프로 조합 가능
capital_chain = prompt_template | llm | output_parser
# invoke가 <- 방향으로 준다면 chain은 -> 방향으로 조합이됨(가독성이 좋음)
# 여기서는 prompt_template에 넣어서 준다면 그 결과가 llm에 들어가고 그 결과가 output_parser에 들어감

In [23]:
# (variable) chain: RunnableSerializable[dict, str]
# chain도 런어블이라 invoke 메서드가 있음
capital_chain.invoke({"country": "France"})

'Paris'

In [24]:
country_prompt = PromptTemplate(
    template="""Guess the name of the country based on the following information:
    {information}
    Return the name of the country only
    """,
    input_variables=["information"]
)

country_chain = country_prompt | llm | output_parser
# country_chain.invoke({"information": "This country is known for its rich history and culture."})
# country_chain.invoke({"information": "This country is known for its beautiful landscapes and delicious cuisine."})

country_chain.invoke({"information": "This country is known for its wine"})


'Italy.'

In [25]:
final_chain = {"country": country_chain} | capital_chain
# country_chain의 결과를 capital_chain에 넣어서 최종 결과를 만들어냄


In [26]:
final_chain.invoke({"information": "This country is known for its wine"})

'Rome'

In [27]:


from langchain_core.runnables import RunnablePassthrough


final_chain2 ={"information": RunnablePassthrough()}| {"country": country_chain} | capital_chain


In [ ]:
final_chain2.invoke("This country is known for its wine")
# 이 정보가 {"information": RunnablePassthrough()}여기로 들어가서 country_chain.invoke({"information": "This country is known for its wine"})이렇게 된다고 이해하면 됨

'Rome'

input_variables가 여러개인 경우

-> RunablePassthrough도 여러개 만들면 됨
final_chain2 ={"information": RunnablePassthrough(), "continent": RunnablePassthrough()}| {"country": country_chain} | capital_chain

이럴땐 평소처럼 튜플 형식으로 줘야댐
final_chain2.invoke({"information":"This country is known for its wine", "continent":"Europe"})
